# Part 3: Clinical Summarization

This notebook implements Part 3 of the in-class Hugging Face activity. It runs a summarization pipeline on the provided messy clinical note and then organizes a critical review of the generated summary.

## Expected Environment

Install the required libraries before running this notebook:

```powershell
uv pip install -r requirements.txt
```

That installs the CUDA 12.6 PyTorch wheels plus the Hugging Face notebook dependencies. If you are using global Python 3.12 instead of a virtual environment, make sure those packages are installed there before running the cells below.

In [ ]:
import torch
from textwrap import fill

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline

In [ ]:
print(f"torch version: {torch.__version__}")

## Source Note

This is the exact clinical note from the assignment. It intentionally contains contradictory details so the summary can be evaluated critically.

In [ ]:
note = """
ED triage: 67-year-old male with HTN and type 2 diabetes presents with
chest pressure radiating to left arm for 2 hours after mowing lawn.
Reports nausea and diaphoresis. Home meds listed as lisinopril and metformin.
Allergies: NKDA.

Nursing addendum 15 min later: patient states he is 64, not 67. Says pain
started yesterday, not today, and now denies radiation, nausea, or sweating.
States he stopped taking all meds 3 months ago. Allergy listed as penicillin
causes rash.

Resident note: wife says he nearly passed out in driveway and was clutching
his chest. Patient denies diabetes but prior chart shows A1c 9.1 last month.
Initial EKG documented as ST elevation in II, III, aVF; repeat note says
"no acute ST changes." Troponin pending.
""".strip()

print(note)

## Run the Summarization Pipeline

The assignment uses `pipeline("summarization")`. The cells below make the download and load step explicit so it is clear which model is being used. This notebook uses a summarization model whose current `main` revision includes `model.safetensors`, and it requests the safetensors weights explicitly.

In [ ]:
model_id = "facebook/bart-large-cnn"

# These calls download the tokenizer/model the first time you run them, then load from cache later.
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id, use_safetensors=True)

print(f"Loaded summarization model: {model_id}")

In [ ]:
summarizer = pipeline("summarization", model=model, tokenizer=tokenizer, framework="pt")

print(summarizer(note, max_length=40))

In [ ]:
summary_result = summarizer(note, max_length=40, min_length=12, do_sample=False)
summary_text = summary_result[0]["summary_text"]

print("Generated summary:\n")
print(fill(summary_text, width=100))

## Key Facts to Compare Against

These facts are copied directly from the note so you can compare the generated summary against the source instead of trusting the model.

In [ ]:
reference_facts = {
    "demographics": [
        "Age is inconsistent: 67 in triage, 64 in nursing addendum.",
        "Patient is male."
    ],
    "symptoms": [
        "Initial triage says chest pressure radiating to the left arm with nausea and diaphoresis.",
        "Later note says pain started yesterday and denies radiation, nausea, and sweating.",
        "Wife reports near-syncope and clutching his chest."
    ],
    "history_and_meds": [
        "History lists hypertension and type 2 diabetes.",
        "Patient later denies diabetes, but prior chart shows A1c 9.1 last month.",
        "Home meds initially listed as lisinopril and metformin.",
        "Patient later says he stopped all meds 3 months ago."
    ],
    "allergies": [
        "Initial triage says NKDA.",
        "Later note says penicillin causes rash."
    ],
    "cardiac_workup": [
        "Initial EKG documented ST elevation in II, III, and aVF.",
        "Repeat note says no acute ST changes.",
        "Troponin is pending."
    ]
}

for category, facts in reference_facts.items():
    print(category.upper())
    for fact in facts:
        print(f"- {fact}")
    print()

## Critical Review Template

Use the generated summary and the reference facts above to answer the assignment prompt.

Focus on three failure modes:

- Hallucinated details: statements in the summary that do not appear in the note.
- Omitted critical facts: high-risk or clinically relevant details that were dropped.
- Overconfidence: wording that presents uncertain or contradictory information as settled fact.

In [ ]:
analysis_template = {
    "hallucinated_details": [
        "No obvious hallucinations in the saved summary."
    ],
    "omitted_critical_facts": [
        "The summary leaves out chest pain radiation, nausea, diaphoresis, the EKG conflict, the allergy conflict, and troponin pending."
    ],
    "overconfidence": [
        "The summary does not clearly show how contradictory the original note is."
    ]
}

analysis_template

## Written Response

**Hallucinated details:**

- No obvious hallucinations in the saved summary.

**Omitted critical facts:**

- The summary leaves out several important details, including the initial chest pain description, nausea/diaphoresis, conflicting EKG findings, allergy conflict, and troponin pending.

**Overconfidence:**

- The summary presents only a few facts and does not clearly show how contradictory the full note is.

**Overall judgment:**

The summary includes some important information, but it leaves out too many clinically important details. It would not be safe to rely on by itself.